<a href="https://colab.research.google.com/github/joaosasantos9-cyber/lab-langchain-evaluation/blob/main/lab_langchain_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [1]:
# STEP 1: Install all dependencies
# After this cell finishes, click RESTART SESSION, then run from the next cell onwards
!pip install langchain-community langchain-openai langchain-huggingface langchain-classic docarray ragas sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

### Example 1

#### Create our QandA application

In [1]:
# STEP 2: Imports (run after RESTART SESSION)
from langchain_classic.chains import RetrievalQA, LLMChain
from langchain_classic.indexes import VectorstoreIndexCreator
from langchain_openai import ChatOpenAI, OpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_community.vectorstores import DocArrayInMemorySearch

In [2]:
# STEP 3: Load API key from Colab Secrets
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [3]:
# STEP 4: Load the CSV file (upload it to Colab first)
file = '/content/OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

In [4]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cpu'})
).from_loaders([loader])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
vectorstore = index.vectorstore
retriever = vectorstore.as_retriever()

In [6]:
llm = ChatOpenAI(temperature=0.0, openai_api_key=OPENAI_API_KEY)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=index.vectorstore.as_retriever(),
    verbose=True,
    chain_type_kwargs={
        "document_separator": "<<<<>>>>>"
    }
)

#### Coming up with test datapoints

In [7]:
data[10]

Document(metadata={'source': '/content/OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [8]:
data[11]

Document(metadata={'source': '/content/OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
             "   Answer: Yes\n"
             "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
             "   Answer: The DownTek collection\n"
             "Query: {query}\n"
             "Answer:"
)

class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)

llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY)

llm_chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    output_parser=AnswerOutputParser()
)

query = "Is the Cozy Comfort Pullover Set available in different colors?"
result = llm_chain.run({"query": query})
print(result)

/tmp/ipykernel_9383/419787952.py:37: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(
/tmp/ipykernel_9383/419787952.py:44: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = llm_chain.run({"query": query})


answer='Yes, it is available in multiple colors such as gray, navy, and black.'


#### LLM-Generated examples

In [10]:
from langchain_classic.evaluation.qa import QAGenerateChain

In [11]:
llm = ChatOpenAI(model="gpt-4o", openai_api_key=OPENAI_API_KEY)
example_gen_chain = QAGenerateChain.from_llm(llm)
llm_chain = LLMChain(llm=llm, prompt=prompt_template)

In [12]:
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

/tmp/ipykernel_9383/3125874183.py:1: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  new_examples = example_gen_chain.apply_and_parse(


In [13]:
d_flattened = [data['qa_pairs'] for data in new_examples]
d_flattened

[{'query': "What features contribute to the comfort and support of the Women's Campside Oxfords, as described in the product document?",
  'answer': "The Women's Campside Oxfords are designed for comfort and support with several features. They include a super-soft canvas for a broken-in feel, a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control, a moderate arch contour of the innersole, and an EVA foam midsole for cushioning and support. Additionally, they have a chain-tread-inspired molded rubber outsole with a modified chain-tread pattern to enhance comfort."},
 {'query': 'What are the specific features and benefits of the Recycled Waterhog Dog Mat, Chevron Weave, as described in the document?',
  'answer': 'The Recycled Waterhog Dog Mat, Chevron Weave, includes several features and benefits as described in the document. It is designed to protect floors from spills and splashing and is constructed from 24 oz. polyester fabric made from 94% recycled materials. T

#### Combine examples

In [14]:
examples += d_flattened

In [15]:
examples[0]

{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
 'answer': 'Yes'}

In [16]:
qa.invoke(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
 'result': 'Yes, the Cozy Comfort Pullover Set does have side pockets.'}

### Manual Evaluation - Fun part

In [17]:
import langchain
langchain.debug = True

result = qa.invoke({
    "query": examples[0]["query"]
})
print(result)



> Entering new RetrievalQA chain...

> Finished chain.
{'query': 'Do the Cozy Comfort Pullover Set have side pockets?', 'result': 'Yes, the Cozy Comfort Pullover Set does have side pockets.'}


In [18]:
langchain.debug = False

### LLM assisted evaluation

In [19]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [20]:
predictions = []
for i, eg in enumerate(examples[:5]):
    result = qa.invoke({"query": eg["query"]})
    pred_dict = {
        'query': eg["query"],
        'answer': eg["answer"],
        'result': result['result'],
        'source_documents': result.get('source_documents', [])
    }
    predictions.append(pred_dict)

for i, eg in enumerate(examples[:5]):
    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real Answer:", predictions[i]['answer'])
    print("Predicted Answer:", predictions[i]['result'])
    print(f"Used {len(predictions[i]['source_documents'])} source documents")
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set have side pockets?
Real Answer: Yes
Predicted Answer: Yes, the Cozy Comfort Pullover Set does have side pockets.
Used 4 source documents

Example 1:
Question: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Used 4 source documents

Example 2:
Question: What features contribute to the comfort and support of the Women's Campside Oxfords, as described in the product document?
Real Answer: The Women's Campside Oxfords are designed for comfort and support with several features. They include a super-soft canvas for a broken-in feel, a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control, a moderate arch contour of the innersole, and an EVA foam midsole for cushioning and support. Additionally, they have a chain-tread-inspired molded rubber outsole with a modified 

### LLM as a Judge

In [21]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, openai_api_key=OPENAI_API_KEY)

eval_prompt = PromptTemplate(
    input_variables=["query", "answer", "result"],
    template="""Score PREDICTED vs ANSWER for QUERY (0-1, higher=better match).

Query: {query}
Answer: {answer}
Predicted: {result}

Score:"""
)

eval_chain = eval_prompt | llm | StrOutputParser()

graded_outputs = []
for i, eg in enumerate(predictions):
    score_raw = eval_chain.invoke({
        "query": predictions[i]['query'],
        "answer": predictions[i]['answer'],
        "result": predictions[i]['result']
    })
    score = float(score_raw.strip())
    graded_outputs.append({'score': score})
    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real:", predictions[i]['answer'])
    print("Predicted:", predictions[i]['result'])
    print("Score:", score, "\n")

Example 0:
Question: Do the Cozy Comfort Pullover Set have side pockets?
Real: Yes
Predicted: Yes, the Cozy Comfort Pullover Set does have side pockets.
Score: 1.0 

Example 1:
Question: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
Real: The DownTek collection
Predicted: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Score: 1.0 

Example 2:
Question: What features contribute to the comfort and support of the Women's Campside Oxfords, as described in the product document?
Real: The Women's Campside Oxfords are designed for comfort and support with several features. They include a super-soft canvas for a broken-in feel, a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control, a moderate arch contour of the innersole, and an EVA foam midsole for cushioning and support. Additionally, they have a chain-tread-inspired molded rubber outsole with a modified chain-tread pattern to enhance comfort.
Predicted: I'm sorr

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [22]:
loader = TextLoader("/content/nyc_text.txt")
index = VectorstoreIndexCreator(
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cpu'})
).from_loaders([loader])

retriever = index.vectorstore.as_retriever()

llm = ChatOpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    return_source_documents=True,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_9383/4274384156.py:2: UserWarning: Using InMemoryVectorStore as the default vectorstore.This memory store won't persist data. You should explicitlyspecify a VectorStore when using VectorstoreIndexCreator
  index = VectorstoreIndexCreator(


In [23]:
question = "How did New York City get its name?"
result = qa_chain.invoke({"query": question})
result["result"]

'New York City was originally named New Amsterdam by Dutch colonists in 1626. When the city came under British control in 1664, it was renamed New York after King Charles II of England granted the lands to his brother, the Duke of York. The city has been continuously named New York since November 1674.'

In [24]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [25]:
all_predictions = []
for example in examples:
    q = example["query"]
    result = qa_chain.invoke({"query": q})
    all_predictions.append({
        "question": q,
        "answer": result['result'],
        "contexts": [doc.page_content for doc in result['source_documents']],
        "ground_truth": example["ground_truths"][0]
    })

print("Predictions generated:", len(all_predictions))
print("\nExample prediction:")
print("Question:", all_predictions[0]['question'])
print("Answer:", all_predictions[0]['answer'])
print("Contexts:", len(all_predictions[0]['contexts']), "documents")
print("Ground Truth:", all_predictions[0]['ground_truth'])

Predictions generated: 5

Example prediction:
Question: What is the population of New York City as of 2020?
Answer: The population of New York City as of 2020 was 8,804,190 residents.
Contexts: 4 documents
Ground Truth: 8,804,190


#### RAGAS Evaluation

In [26]:
import os
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset

dataset_all = Dataset.from_list(all_predictions)

print("=" * 50)
print("RAGAS Evaluation - Faithfulness")
print("=" * 50)
f = evaluate(dataset_all, metrics=[faithfulness])
print(f)

print("\n" + "=" * 50)
print("RAGAS Evaluation - Context Recall")
print("=" * 50)
r = evaluate(dataset_all, metrics=[context_recall])
print(r)

print("\n" + "=" * 50)
print("RAGAS Evaluation - Combined")
print("=" * 50)
results = evaluate(dataset_all, metrics=[faithfulness, context_recall])
print(results)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_9383/3559150580.py:5: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
/tmp/ipykernel_9383/3559150580.py:5: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  fr

RAGAS Evaluation - Faithfulness


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'faithfulness': 1.0000}

RAGAS Evaluation - Context Recall


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'context_recall': 0.7333}

RAGAS Evaluation - Combined


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'context_recall': 0.8000}


#### RAGAS with EvaluatorChain

In [28]:
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset
import os
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# Section 1: Single example
query = examples[0]["query"]
result = qa_chain.invoke({"query": query})

single_dataset = Dataset.from_list([{
    "question": query,
    "answer": result["result"],
    "contexts": [doc.page_content for doc in result["source_documents"]],
    "ground_truth": examples[0]["ground_truths"][0]
}])

single_result = evaluate(single_dataset, metrics=[faithfulness, context_recall])
print(single_result)

/tmp/ipykernel_9383/1798191955.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
/tmp/ipykernel_9383/1798191955.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'context_recall': 1.0000}


In [30]:
# Section 2: Batch evaluation
dataset_all = Dataset.from_list(all_predictions)
results = evaluate(dataset_all, metrics=[faithfulness, context_recall])
print(results)

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'context_recall': 0.7333}
